In [16]:
code = """
import os
import sys
import subprocess

def get_user():
    return input()

def wrapper():
    return get_user()

x = get_user()
cursor.execute(x)

user = input()
query = "SELECT " + user
cursor.execute(query)

q = f"SELECT {user}"
db.execute(q)

user = input()
cursor.execute("SELECT * FROM users WHERE id=%s", (user,))

safe = int(input())
cursor.execute(safe)

a = input()
b = a
c = b
d = "SELECT " + c
cursor.execute(d)

z = wrapper()
cursor.execute(z)

env = os.environ
cursor.execute(env)

arg = sys.argv[1]
cursor.execute(arg)

user = input()
html = "<h1>" + user
response.write(html)

name = input()
render(name)

cmd = input()
os.system(cmd)

cmd2 = input()
subprocess.run(cmd2)

x = input()
eval(x)
"""

In [10]:
tree = ast.parse(code3)

analyzer = Analyzer()
analyzer.visit(tree)

if not analyzer.vulns:
    print("No vulnerability detected")
else: 
     for v in analyzer.vulns:
      print("==== VULNERABILITY DETECTED ====")
      print("TYPE      :", v["type"])
      print("VARIABLE  :", v["var"])
      print("LINE      :", v["lineno"])
      print("SOURCE    :", v["source"])
      print("SINK      :", v["sink"])
      print("================================\n")

No vulnerability detected


In [17]:
code2 = """
def get_user():
    return input()

def wrapper():
    return get_user()

x = get_user()
cursor.execute(x)
"""

In [6]:
code1 = """
import os
import sys



k="piyush"
query = "SELECT " + k
cursor.execute(query)
"""


In [13]:
code3="""
import os

x = input()        

def f():
    x = "safe"   

os.system(x)      
"""

In [ ]:
import ast

class Analyzer(ast.NodeVisitor):

    def __init__(self):
        self.scopes = [{}]                 
        self.globals_declared = [set()]
        self.sources = []
        self.func_returns = {}
        self.vulns = []
        self.sanitizers = ["escape","sanitize","quote","int","str","html_escape"]

    def lookup(self,name):
        for s in reversed(self.scopes):
            if name in s:
                return s[name]             
        return None

    def target_scope(self,name):
        if name in self.globals_declared[-1]:
            return self.scopes[0]
        return self.scopes[-1]

    def is_tainted(self,node):
        return isinstance(node, ast.Name) and self.lookup(node.id) is not None

    def mark(self,node,src):
        if isinstance(node, ast.Name):
            self.target_scope(node.id)[node.id] = src

    def unmark(self,node):
        if isinstance(node, ast.Name):
            self.target_scope(node.id)[node.id] = None

    def is_source(self,node):

        if isinstance(node,ast.Call):

            if isinstance(node.func,ast.Name):
                if node.func.id in ["input"]:
                    return "user_input"

            if isinstance(node.func,ast.Attribute):
                if node.func.attr in ["getenv","read","recv","json","form","args"]:
                    return "external_input"

        if isinstance(node,ast.Attribute):
            if isinstance(node.value,ast.Name):
                if node.value.id=="request":
                    return "web_input"

                if node.value.id=="os" and node.attr=="environ":
                    return "env"

        if isinstance(node,ast.Subscript):
            if isinstance(node.value,ast.Attribute):
                if isinstance(node.value.value,ast.Name):
                    if node.value.value.id=="sys" and node.value.attr=="argv":
                        return "argv"

        return None

    def get_sink_type(self,node):

        if isinstance(node,ast.Call):

            if isinstance(node.func,ast.Attribute):
                f = node.func.attr

                if f in ["execute","executemany","query","raw","to_sql"]:
                    return "SQLi"

                if f in ["write","send","render","render_template"]:
                    return "XSS"

                if f in ["system","popen","run","call"]:
                    return "OS_CMD"

            if isinstance(node.func,ast.Name):
                f = node.func.id

                if f in ["eval","exec"]:
                    return "OS_CMD"

        return None

    def is_sanitized(self,node):

        if isinstance(node,ast.Call):
            if isinstance(node.func,ast.Name):
                if node.func.id in self.sanitizers:
                    return True

        return False

    def visit_Global(self,node):
        self.globals_declared[-1].update(node.names)

    def visit_FunctionDef(self,node):

        for stmt in node.body:
            if isinstance(stmt,ast.Return):
                if self.is_source(stmt.value):
                    self.func_returns[node.name]=True

        # naya scope
        self.scopes.append({})
        self.globals_declared.append(set())

        a = node.args
        params = a.posonlyargs + a.args + a.kwonlyargs
        if a.vararg:
            params.append(a.vararg)
        if a.kwarg:
            params.append(a.kwarg)
        for p in params:
            self.scopes[-1][p.arg] = None   # parameter = local + clean

        self.generic_visit(node)

        # scope khatam
        self.scopes.pop()
        self.globals_declared.pop()

    visit_AsyncFunctionDef = visit_FunctionDef

    def visit_Assign(self,node):

        src = None

        s = self.is_source(node.value)
        if s:
            src = node
            self.sources.append(node)

        elif isinstance(node.value,ast.Name) and self.is_tainted(node.value):
            src = self.lookup(node.value.id)

        elif isinstance(node.value,ast.Call):

            if self.is_sanitized(node.value):
                src = None

            elif isinstance(node.value.func,ast.Name):
                if node.value.func.id in self.func_returns:
                    src = node

            for a in node.value.args:
                if isinstance(a,ast.Name) and self.is_tainted(a):
                    src = self.lookup(a.id)

        elif isinstance(node.value,ast.BinOp):

            if isinstance(node.value.left,ast.Name) and self.is_tainted(node.value.left):
                src = self.lookup(node.value.left.id)

            if isinstance(node.value.right,ast.Name) and self.is_tainted(node.value.right):
                src = self.lookup(node.value.right.id)

        elif isinstance(node.value,ast.JoinedStr):

            for v in node.value.values:
                if isinstance(v,ast.FormattedValue):
                    if isinstance(v.value,ast.Name) and self.is_tainted(v.value):
                        src = self.lookup(v.value.id)

        for t in node.targets:
            self.unmark(t)

        if src:
            for t in node.targets:
                self.mark(t,src)

        self.generic_visit(node)

    def visit_Call(self,node):

        sink_type = self.get_sink_type(node)

        if sink_type:

            for arg in node.args:
                if isinstance(arg,ast.Name) and self.is_tainted(arg):

                    src = self.lookup(arg.id)

                    self.vulns.append({
                        "type": sink_type,
                        "source": ast.dump(src),
                        "sink": ast.dump(node),
                        "var": arg.id,
                        "lineno": node.lineno
                    })

            for kw in node.keywords:
                if isinstance(kw.value,ast.Name) and self.is_tainted(kw.value):

                    src = self.lookup(kw.value.id)

                    self.vulns.append({
                        "type": sink_type,
                        "source": ast.dump(src),
                        "sink": ast.dump(node),
                        "var": kw.value.id,
                        "lineno": node.lineno
                    })

        self.generic_visit(node)

In [22]:
tree = ast.parse(code4)

analyzer = Analyzer()
analyzer.visit(tree)

if not analyzer.vulns:
    print("No vulnerability detected")
else: 
     for v in analyzer.vulns:
      print("==== VULNERABILITY DETECTED ====")
      print("TYPE      :", v["type"])
      print("VARIABLE  :", v["var"])
      print("LINE      :", v["lineno"])
      print("SOURCE    :", v["source"])
      print("SINK      :", v["sink"])
      print("================================\n")

==== VULNERABILITY DETECTED ====
TYPE      : OS_CMD
VARIABLE  : x
LINE      : 7
SOURCE    : Assign(targets=[Name(id='x', ctx=Store())], value=Call(func=Name(id='input', ctx=Load()), args=[], keywords=[]))
SINK      : Call(func=Attribute(value=Name(id='os', ctx=Load()), attr='system', ctx=Load()), args=[Name(id='x', ctx=Load())], keywords=[])



In [20]:
code4 = '''
import os

x = input()
def f():
    x = "safe"
os.system(x)

y = "ls"
def g():
    y = input()
os.system(y)

def h(data):
    os.system(data)
'''